# Module 1 Lab — Agentic AI & Advanced Analytics in Finance
### ISBMS · PGDM 2025–27 · Session 1

**Nothing to install on your laptop.** This runs in your browser on Google's machine.

Four steps, about 45 minutes:

1. Look up any company's share price and revenue
2. Find the mistake the data makes, and catch it
3. Pull numbers out of a filing with plain rules, and watch the rules break
4. Watch an agent loop do a real back-office job

Run each cell with **Shift + Enter**. If a cell asks you a question, type the answer and press Enter.


---
## Step 0 — Set up (20 seconds)

`yfinance` is a free library that reads Yahoo Finance. Run this once.


In [ ]:
!pip install yfinance -q
import yfinance as yf
print('Ready. yfinance version:', yf.__version__)


---
## Step 1 — Look up a company

Type a company name when the cell asks. Try **Reliance** first, then run it again for **Infosys**.


In [ ]:
import yfinance as yf

KNOWN = {'reliance': 'RELIANCE.NS', 'infosys': 'INFY.NS', 'tcs': 'TCS.NS',
         'hdfc bank': 'HDFCBANK.NS', 'itc': 'ITC.NS', 'apple': 'AAPL'}

def find_ticker(name):
    """Turn a company name into a Yahoo Finance ticker."""
    if name.lower() in KNOWN:
        return KNOWN[name.lower()]
    if '.' in name or name.isupper():      # the user typed a ticker already
        return name
    hits = yf.Search(name).quotes           # ask Yahoo to search
    indian = [h for h in hits if h.get('exchange') == 'NSI']
    return (indian or hits)[0]['symbol']    # prefer the NSE listing

def look_up(name):
    symbol = find_ticker(name)
    t = yf.Ticker(symbol)
    info = t.info
    fin = t.financials

    revenue, period = None, None
    if fin is not None and not fin.empty and 'Total Revenue' in fin.index:
        column = fin.columns[0]                 # most recent year
        revenue = float(fin.loc['Total Revenue', column])
        period = str(column)[:10]

    return {
        'name': info.get('longName'),
        'ticker': symbol,
        'price': info.get('regularMarketPrice'),
        'price_currency': info.get('currency'),
        'revenue': revenue,
        'revenue_currency': info.get('financialCurrency'),
        'period_ending': period,
    }

company = input('Company name: ')
result = look_up(company)

for key, value in result.items():
    print(f'{key:>18} : {value}')


---
## Step 2 — Now find the mistake

Run the cell below for **Infosys**.

The share price comes back in rupees. The revenue does not. Yahoo reports Infosys' statements
in **US dollars**, because that is how Infosys files for its overseas listing. The two fields
disagree, and nothing warns you.

This is not a made-up example. The app built in class printed Infosys revenue as **INR 20.16B**.
The real figure is about **₹1.6 lakh crore**, which is ₹1.6T. Wrong by roughly eighty times,
on a clean screen, with no error message.

One line of code catches it.


In [ ]:
def check(result):
    """Four checks, in the order a desk applies them."""
    problems = []

    if result['price'] is None:
        problems.append('No price came back at all.')
    if result['revenue'] is None:
        problems.append('No revenue came back at all.')

    # THE IMPORTANT ONE
    if result['price_currency'] != result['revenue_currency']:
        problems.append(
            f"Currency mismatch: price is in {result['price_currency']}, "
            f"revenue is in {result['revenue_currency']}. "
            'Do NOT put these two numbers in the same table.')

    if result['period_ending'] is None:
        problems.append('No period stated, so you cannot say which year this is.')

    if problems:
        print('PROBLEMS FOUND')
        for p in problems:
            print('  -', p)
    else:
        print('All four checks passed.')
        print(f"  {result['name']} ({result['ticker']})")
        print(f"  Price   : {result['price_currency']} {result['price']:,.2f}")
        print(f"  Revenue : {result['revenue_currency']} {result['revenue']:,.0f} "
              f"for the year to {result['period_ending']}")

for name in ['Reliance', 'Infosys']:
    print('=' * 60)
    print(name.upper())
    check(look_up(name))


**Write this down.** The app did not crash. The number was simply wrong, and it looked fine.
Your value on a desk is that you are the one who checks.


---
## Step 3 — Read a filing with plain rules

No AI here. Six labels, ordinary Python, and two accounting checks. Run it and read the result.


In [ ]:
import re

FILING = '''
MERIDIAN CHEMICALS LIMITED
Consolidated Statement of Profit and Loss   (Rs in crore)

Revenue from operations                              62,480
Other income                                          1,905
Total income                                         64,385
Total expenses                                       52,140
Profit before share of associates and tax            12,245
Share of profit of associates                           330
Profit before tax                                    12,575
Tax expense                                           3,145
Profit for the year                                   9,430
'''

LABELS = {
    'revenue_from_operations': 'revenue from operations',
    'other_income': 'other income',
    'total_expenses': 'total expenses',
    'profit_before_tax': 'profit before tax',
    'tax_expense': 'tax expense',
    'profit_after_tax': 'profit for the year',
}

def grab(text, label):
    """Find the label, then take the first number printed after it."""
    m = re.search(re.escape(label) + r'[^\d\n]*([\d,]+)', text, re.IGNORECASE)
    return int(m.group(1).replace(',', '')) if m else None

items = {field: grab(FILING, label) for field, label in LABELS.items()}
for field, value in items.items():
    print(f'{field:>28} : {value:,}' if value else f'{field:>28} : not found')

print()
expected_pbt = items['revenue_from_operations'] + items['other_income'] - items['total_expenses']
print('CHECK 1  revenue + other income - expenses = PBT ?')
print(f'         {expected_pbt:,} calculated  vs  {items["profit_before_tax"]:,} printed',
      '-> PASS' if expected_pbt == items['profit_before_tax'] else '-> FAIL')

expected_pat = items['profit_before_tax'] - items['tax_expense']
print('CHECK 2  PBT - tax = PAT ?')
print(f'         {expected_pat:,} calculated  vs  {items["profit_after_tax"]:,} printed',
      '-> PASS' if expected_pat == items['profit_after_tax'] else '-> FAIL')


**Check 1 fails, and the numbers are all correct.**

This company adds *share of profit of associates* (330) between total expenses and profit before tax.
The rule was too simple, not the data. Telling those two apart is the skill.

Now try it yourself: replace the text in `FILING` with a P&L from any annual report PDF and run it again.
Most of the time the labels will not match and the rules will find nothing. That is the point.
Rules break on every new format. A model does not.


---
## Step 4 — Watch an agent loop work

A fund's NAV does not match the custodian's. The loop observes, plans, acts and reflects until
the gap is explained. There is no AI in this cell. Read the loop; that is the exercise.


In [ ]:
FUND_NAV, CUSTODIAN_NAV, UNITS = 104.40, 104.82, 1_000_000
BONUS_SHARES, EX_BONUS_PRICE = 1_200, 350.0     # a 1:1 bonus we have not recorded
STEP_BUDGET = 4                                  # the brake

gap = round((CUSTODIAN_NAV - FUND_NAV) * UNITS, 2)
print(f'OBSERVE  custodian is higher by Rs {CUSTODIAN_NAV - FUND_NAV:.2f} per unit',
      f'= Rs {gap:,.0f} in total')

plan = ['check today trades', 'check corporate actions', 'check the FX rate']
print('PLAN    ', ' | '.join(plan))

explained = 0.0
for step, task in enumerate(plan, 1):
    if step > STEP_BUDGET:
        print('STOP     step budget reached, handing to a human'); break

    if task == 'check corporate actions':
        found = BONUS_SHARES * EX_BONUS_PRICE
        explained += found
        print(f'ACT {step}    {task} -> {BONUS_SHARES:,} bonus shares not recorded',
              f'x Rs {EX_BONUS_PRICE:,.0f} = Rs {found:,.0f}')
    else:
        print(f'ACT {step}    {task} -> nothing found')

    remaining = gap - explained
    print(f'REFLECT  explained Rs {explained:,.0f}, still unexplained Rs {remaining:,.0f}')
    if abs(remaining) < 1:
        print()
        print('GOAL MET  The whole difference is the unprocessed 1:1 bonus.')
        print('          Draft note goes to the fund accountant. A HUMAN SIGNS IT.')
        break
else:
    print('GOAL NOT MET  Escalate. Do not invent a reason.')


**Change one thing and run it again:** set `BONUS_SHARES = 0`.
The loop now explains nothing and escalates, instead of inventing an answer. That behaviour is
the difference between an agent you can put in production and one you cannot.


---
## What to hand in before Session 2

One page. Four lines:

1. Three companies you looked up, with price, revenue and the currency of each.
2. One number that was wrong, and the check that caught it.
3. Your one-line answer to: why did Check 1 fail in Step 3 even though every number was right?
4. What the agent loop did when you set the bonus shares to zero.

Bring one annual report PDF to the next class. Module 2 turns it into a searchable knowledge base.
